In [ ]:
import os
import cv2
import numpy as np
import torch
import torchvision.models as models

# ======================
# Device
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Load ResNet152 (feature extractor)
# ======================
model = models.resnet152(pretrained=True)

# remove classification layer
model = torch.nn.Sequential(*list(model.children())[:-1])

model = model.to(device)
model.eval()

In [ ]:
def extract_resnet_features(folder_path):
    features_list = []
    labels = []

    classes = sorted(os.listdir(folder_path))

    with torch.no_grad():
        for class_name in classes:
            class_path = os.path.join(folder_path, class_name)

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                img = cv2.imread(img_path)
                if img is None:
                    continue

                # resize
                img = cv2.resize(img, (224, 224))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # normalize to [0,1]
                img = img / 255.0

                img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)

                # ImageNet normalization (VERY IMPORTANT)
                img = (img - torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)) / \
                      torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

                img = img.unsqueeze(0).to(device)

                # extract features
                feat = model(img)

                # flatten (ResNet outputs [1,2048,1,1])
                feat = feat.view(feat.size(0), -1)

                features_list.append(feat.cpu().numpy().squeeze())
                labels.append(class_name)

    return np.array(features_list), np.array(labels)

In [ ]:
X_train_resnet, y_train_resnet = extract_resnet_features(r"C:\Users\Karim\Contacts\Desktop\split_data\train_AUGMENTED")
X_test_resnet, y_test_resnet   = extract_resnet_features(r"C:\Users\Karim\Contacts\Desktop\split_data\test")

print(X_train_resnet.shape)
print(X_test_resnet.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_resnet = le.fit_transform(y_train_resnet)
y_test_resnet = le.transform(y_test_resnet)

In [ ]:
import numpy as np

dataset_name = "AC"

# Save TRAIN
np.save(f"{dataset_name}_X_train_resnet.npy", X_train_resnet)
np.save(f"{dataset_name}_y_train_resnet.npy", y_train_resnet)

# Save TEST
np.save(f"{dataset_name}_X_test_resent.npy", X_test_resnet)
np.save(f"{dataset_name}_y_test_resnet.npy", y_test_resnet)

print("✅ Train and Test features saved successfully!")